# Day3_18 · 여러 기상값을 줄이고 묶기

울산의 기온·강수·습도·바람·일사량을 두 축으로 요약하고 비슷한 날을 묶습니다.

## 이 노트북에서 확인할 내용

- 단위가 다른 수치열을 표준화할 수 있습니다.
- PCA의 두 축과 원래 기상열의 관계를 확인할 수 있습니다.
- K-means 집단을 원래 단위의 평균표로 설명할 수 있습니다.

## 실행 방법

위에서 아래로 코드 셀을 한 개씩 실행하고 바로 다음 결과를 확인합니다. 수정 안내가 있는 곳에서는 표시된 값 하나만 바꾼 뒤 결과를 비교합니다.

### D3B-18-01 · 경로 도구 불러오기

In [ ]:
from pathlib import Path

### D3B-18-02 · 현재 폴더 확인

In [ ]:
current_folder = Path.cwd()

In [ ]:
current_folder

### D3B-18-03 · 데이터 폴더 후보 만들기

In [ ]:
data_candidates = [
    current_folder / 'data',
    current_folder.parent / 'data',
]

In [ ]:
data_candidates

### D3B-18-04 · 데이터 폴더 선택

In [ ]:
DATA_DIR = next(folder for folder in data_candidates if folder.exists())

In [ ]:
DATA_DIR

### D3B-18-05 · pandas 불러오기

In [ ]:
import pandas as pd

### D3B-18-06 · 그림 도구 불러오기

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

### D3B-18-07 · 울산 일별 기상자료 읽기

In [ ]:
weather = pd.read_csv(DATA_DIR / 'weather_daily.csv')

In [ ]:
weather.head()

### D3B-18-08 · 날짜를 날짜 형식으로 바꾸기

In [ ]:
weather['date'] = pd.to_datetime(weather['date'])

In [ ]:
weather[['date']].head()

### D3B-18-09 · 비교할 기상 열 정하기

In [ ]:
weather_columns = ['temperature_mean_c', 'precipitation_mm', 'humidity_percent', 'wind_speed_ms', 'solar_energy_kwh_m2']

In [ ]:
pd.Series(weather_columns)

### D3B-18-10 · 기상 열 요약 확인

In [ ]:
weather[weather_columns].describe().round(2)

## 단위가 다른 수치를 같은 범위로 바꾸기

기온, 강수량, 습도, 바람, 일사량은 단위가 다릅니다. 각 열의 평균을 0, 표준편차를 1에 가깝게 바꾼 뒤 여러 열을 함께 비교합니다.

### D3B-18-11 · 표준화 도구 불러오기

In [ ]:
from sklearn.preprocessing import StandardScaler

### D3B-18-12 · 기상 열의 변환 규칙 익히기

In [ ]:
weather_scaler = StandardScaler().fit(weather[weather_columns])

In [ ]:
weather_scaler

### D3B-18-13 · 기상 열 표준화하기

In [ ]:
weather_scaled = pd.DataFrame(weather_scaler.transform(weather[weather_columns]), columns=weather_columns, index=weather.index)

In [ ]:
weather_scaled.head().round(3)

### D3B-18-14 · 표준화한 열의 평균과 표준편차 확인

In [ ]:
scaled_check = weather_scaled.agg(['mean', 'std'])

In [ ]:
scaled_check.round(2)

## 여러 기상열을 두 개의 새 축으로 요약하기

여러 열의 변화가 많이 담기도록 새로운 축을 만듭니다. 전문가들은 이 방법을 주성분분석, PCA라고 부릅니다.

### D3B-18-15 · PCA 도구 불러오기

In [ ]:
from sklearn.decomposition import PCA

### D3B-18-16 · 두 개의 새 축 만들기

In [ ]:
weather_pca = PCA(n_components=2).fit(weather_scaled)

In [ ]:
weather_pca

### D3B-18-17 · 각 축이 담은 변동 비율 확인

In [ ]:
explained_variance = pd.Series(weather_pca.explained_variance_ratio_, index=['component_1', 'component_2'])

In [ ]:
explained_variance.round(3)

### D3B-18-18 · 날짜별 새 좌표 계산

In [ ]:
component_values = weather_pca.transform(weather_scaled)
weather_components = pd.DataFrame(component_values, columns=['component_1', 'component_2'])

In [ ]:
weather_components.head().round(3)

### D3B-18-19 · 날짜와 월을 새 좌표에 추가

In [ ]:
weather_components['date'] = weather['date']
weather_components['month'] = weather['date'].dt.month

In [ ]:
weather_components.head()

### D3B-18-20 · 월별 색으로 새 좌표 그리기

In [ ]:
sns.scatterplot(data=weather_components, x='component_1', y='component_2', hue='month', palette='turbo', alpha=0.75)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()

### D3B-18-21 · 새 축과 원래 기상열의 연결표 만들기

In [ ]:
loading_table = pd.DataFrame(weather_pca.components_.T, index=weather_columns, columns=['component_1', 'component_2'])

In [ ]:
loading_table.round(3)

### D3B-18-22 · 연결표를 색으로 보기

In [ ]:
sns.heatmap(loading_table, annot=True, cmap='vlag', center=0)
plt.title('How original weather columns form two components')
plt.show()

PCA의 새 축은 원래 열의 가중합입니다. 부호 자체보다 어떤 열이 같은 방향 또는 반대 방향으로 큰 값을 갖는지 확인합니다.

## 비슷한 날을 자동으로 묶기

정답 열 없이 표준화된 기상값이 가까운 날을 묶습니다. 전문가들은 이와 같은 방법을 군집분석이라고 부릅니다.

### D3B-18-23 · K-means 도구 불러오기

In [ ]:
from sklearn.cluster import KMeans

### D3B-18-24 · 네 개의 기상 집단 만들기

In [ ]:
weather_cluster_model = KMeans(n_clusters=4, random_state=42, n_init=20).fit(weather_scaled)

In [ ]:
weather_cluster_model

### D3B-18-25 · 각 날짜에 집단 번호 붙이기

In [ ]:
weather_clustered = weather.copy()
weather_clustered['cluster'] = weather_cluster_model.labels_

In [ ]:
weather_clustered[['date', 'cluster'] + weather_columns].head()

### D3B-18-26 · 집단별 날짜 수 확인

In [ ]:
weather_clustered['cluster'].value_counts().sort_index()

### D3B-18-27 · 집단별 기상 평균표 만들기

In [ ]:
cluster_summary = weather_clustered.groupby('cluster')[weather_columns].mean()

In [ ]:
cluster_summary.round(2)

### D3B-18-28 · 새 좌표에 집단 번호 추가

In [ ]:
weather_components['cluster'] = weather_cluster_model.labels_

In [ ]:
weather_components.head()

### D3B-18-29 · 집단별 색으로 새 좌표 그리기

In [ ]:
sns.scatterplot(data=weather_components, x='component_1', y='component_2', hue='cluster', palette='Set2', alpha=0.8)
plt.show()

### D3B-18-30 · 집단과 월의 교차표 만들기

In [ ]:
cluster_month_table = pd.crosstab(weather_clustered['cluster'], weather_clustered['date'].dt.month)

In [ ]:
cluster_month_table

### D3B-18-31 · 집단과 월의 교차표를 색으로 보기

In [ ]:
sns.heatmap(cluster_month_table, cmap='YlGnBu')
plt.xlabel('Month')
plt.ylabel('Cluster')
plt.show()

### D3B-18-32 · 집단 수 비교 도구 불러오기

In [ ]:
from sklearn.metrics import silhouette_score

### D3B-18-33 · 집단 수 2~6의 분리 정도 계산

In [ ]:
cluster_score_table = pd.DataFrame([
    {'clusters': count, 'silhouette': silhouette_score(weather_scaled, KMeans(n_clusters=count, random_state=42, n_init=20).fit_predict(weather_scaled))}
    for count in range(2, 7)
])

In [ ]:
cluster_score_table.round(3)

### D3B-18-34 · 집단 수와 분리 정도 그리기

In [ ]:
sns.lineplot(data=cluster_score_table, x='clusters', y='silhouette', marker='o')
plt.ylim(0, 1)
plt.show()

### D3B-18-35 · 확인할 집단 번호 정하기

In [ ]:
# 0을 1, 2 또는 3으로 바꾸어 보세요.
selected_cluster = 0

In [ ]:
selected_cluster

### D3B-18-36 · 선택한 집단의 날짜 확인

In [ ]:
selected_cluster_days = weather_clustered.loc[weather_clustered['cluster'] == selected_cluster]

In [ ]:
selected_cluster_days[['date'] + weather_columns].head(15).round(2)

집단 번호에는 순위나 좋고 나쁨의 뜻이 없습니다. 집단의 이름은 평균 기상표와 날짜 분포를 확인한 뒤 연구자가 붙입니다.

### D3B-18-CODEX · Codex에 확장 분석 요청하기

In [ ]:
codex_request = '''weather_clustered와 cluster_summary를 사용하세요. 기존 셀은 수정하지 마세요. 각 cluster의 기온·강수·습도 평균을 비교하는 heatmap을 새 셀로 제안하고, 각 집단에 자료 근거가 드러나는 짧은 이름을 붙여 주세요. 집단 번호를 순위로 해석하지 마세요.'''.strip()

In [ ]:
codex_request

### D3B-18-CHECK · 제안받은 코드를 실행하기 전에 확인하기

In [ ]:
codex_checklist = pd.Series(['원래 단위의 평균표를 사용하는가?', '집단 번호를 좋고 나쁨으로 해석하지 않는가?', '날짜나 월 분포로 이름을 확인하는가?'], name='확인할 내용')

In [ ]:
codex_checklist